# intersect_qef old/ref/new JIT comparison

This notebook JIT-compiles temporary bindings for `intersect_qef_old`, `intersect_qef_ref`, `intersect_qef`, and a CPU-only `intersect_qef` wrapper. It then uses `notebooks/test.glb` to compare occupancy, QEF outputs, timing, and CUDA peak memory.

Run this notebook from the `symm-enforce` conda environment.

In [6]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

import torch
from torch.utils.cpp_extension import load

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)
torch.set_grad_enabled(False)

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

# GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '128'))
GRID_SIZE = 512
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)

repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000


## JIT compile temporary extension

The binding source is generated inside this notebook into a temporary directory. No project source file is modified here.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_intersect_jit_'))
binding_path = build_dir / 'binding.cpp'

binding_cpp = f'''
#include <torch/extension.h>
#include <Eigen/Dense>
#include <unordered_map>
#include <vector>
#include <cmath>
#include "{(ROOT / 'src/convert/flexible_dual_grid.cpp').as_posix()}"
#include "{(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/api.h').as_posix()}"

namespace py = pybind11;

namespace {{

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>
intersect_qef_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    int64_t chunk_triangles) {{
    (void)chunk_triangles;
    const int64_t T = triangles.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();

    std::vector<Eigen::Vector3f> tri_vec;
    tri_vec.reserve(static_cast<size_t>(T) * 3);
    for (int64_t t = 0; t < T; ++t) {{
        tri_vec.emplace_back(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        tri_vec.emplace_back(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        tri_vec.emplace_back(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
    }}

    std::unordered_map<VoxelCoord, size_t> hash_table;
    std::vector<int3> voxels;
    std::vector<Eigen::Vector3f> means;
    std::vector<float> cnt;
    std::vector<bool3> intersected;
    std::vector<Eigen::Matrix4f> qefs;
    intersect_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        tri_vec,
        hash_table,
        voxels,
        means,
        cnt,
        intersected,
        qefs);

    const int64_t N = static_cast<int64_t>(voxels.size());
    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(torch::kCPU);
    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU);
    auto opts_bool = torch::TensorOptions().dtype(torch::kBool).device(torch::kCPU);
    auto out_voxels = torch::empty({{N, 3}}, opts_i32);
    auto out_mean = torch::empty({{N, 3}}, opts_f32);
    auto out_cnt = torch::empty({{N}}, opts_f32);
    auto out_intersected = torch::empty({{N, 3}}, opts_bool);
    auto out_qefs = torch::empty({{N, 10}}, opts_f32);

    int32_t* vox_ptr = out_voxels.data_ptr<int32_t>();
    float* mean_ptr = out_mean.data_ptr<float>();
    float* cnt_ptr = out_cnt.data_ptr<float>();
    bool* inter_ptr = out_intersected.data_ptr<bool>();
    float* qef_ptr = out_qefs.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {{
        vox_ptr[3 * i + 0] = voxels[i].x;
        vox_ptr[3 * i + 1] = voxels[i].y;
        vox_ptr[3 * i + 2] = voxels[i].z;
        mean_ptr[3 * i + 0] = means[i].x();
        mean_ptr[3 * i + 1] = means[i].y();
        mean_ptr[3 * i + 2] = means[i].z();
        cnt_ptr[i] = cnt[i];
        inter_ptr[3 * i + 0] = intersected[i].x;
        inter_ptr[3 * i + 1] = intersected[i].y;
        inter_ptr[3 * i + 2] = intersected[i].z;
        const Eigen::Matrix4f& Q = qefs[i];
        qef_ptr[10 * i + 0] = Q(0, 0);
        qef_ptr[10 * i + 1] = Q(0, 1);
        qef_ptr[10 * i + 2] = Q(0, 2);
        qef_ptr[10 * i + 3] = Q(0, 3);
        qef_ptr[10 * i + 4] = Q(1, 1);
        qef_ptr[10 * i + 5] = Q(1, 2);
        qef_ptr[10 * i + 6] = Q(1, 3);
        qef_ptr[10 * i + 7] = Q(2, 2);
        qef_ptr[10 * i + 8] = Q(2, 3);
        qef_ptr[10 * i + 9] = Q(3, 3);
    }}
    return std::make_tuple(out_voxels, out_mean, out_cnt, out_intersected, out_qefs);
}}

torch::Tensor intersect_occ_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    int64_t chunk_triangles) {{
    return std::get<0>(intersect_qef_cpu_only(triangles, voxel_size, grid_range, chunk_triangles));
}}

}} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {{
    m.def("intersect_qef_cpu", &intersect_qef_cpu_only);
    m.def("intersect_occ_cpu", &intersect_occ_cpu_only);
    m.def("intersect_qef_old", &o_voxel::fdg::intersect_qef_old);
    m.def("intersect_occ_old", &o_voxel::fdg::intersect_occ_old);
    m.def("intersect_qef_ref", &o_voxel::fdg::intersect_qef_ref);
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("intersect_occ", &o_voxel::fdg::intersect_occ);
}}
'''

binding_path.write_text(binding_cpp)

sources = [
    str(binding_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
]

ext = load(
    name=f'fdg_intersect_jit_{os.getpid()}',
    sources=sources,
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17'],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)

Using /home/quanta/.cache/torch_extensions/py310_cu124 as PyTorch extensions root...
Creating extension directory /home/quanta/.cache/torch_extensions/py310_cu124/fdg_intersect_jit_477704...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/quanta/.cache/torch_extensions/py310_cu124/fdg_intersect_jit_477704/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_intersect_jit_477704...
Using envvar MAX_JOBS (32) as the number of workers...


[1/5] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_intersect_jit_477704 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -std=c++17 -c /tmp/fdg_intersect_jit_ut9cy8b1/bi

Loading extension module fdg_intersect_jit_477704...


## Load `notebooks/test.glb` and build lower-level triangle input

In [7]:
import numpy as np
import trimesh

glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).contiguous()
triangles_cpu = vertices_cpu[faces_cpu].reshape(-1, 9).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('vertices:', tuple(vertices_cpu.shape), 'faces:', tuple(faces_cpu.shape), 'triangles:', tuple(triangles_cpu.shape))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())

vertices: (268018, 3) faces: (280333, 3) triangles: (280333, 9)
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


Important: the GPU functions take CUDA `triangles`, but `voxel_size` and `grid_range` are passed as CPU tensors because the current C++ reads those scalar tensors on the host.

## Correctness helpers

In [8]:
def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])

def canonicalize_qef(output):
    voxels, mean_sum, cnt, intersected, qefs = output
    voxels = voxels.detach().cpu().contiguous()
    mean_sum = mean_sum.detach().cpu().float().contiguous()
    cnt = cnt.detach().cpu().float().contiguous()
    intersected = intersected.detach().cpu().bool().contiguous()
    qefs = qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels[order],
        'mean_sum': mean_sum[order],
        'cnt': cnt[order],
        'intersected': intersected[order],
        'qefs': qefs[order],
        'unique': bool(unique),
    }

def canonicalize_occ(voxels):
    voxels = voxels.detach().cpu().contiguous()
    keys = voxel_key(voxels)
    order = torch.argsort(keys)
    return {'keys': keys[order], 'voxels': voxels[order]}

def compare_occ(base, other):
    a = base['keys']
    b = other['keys']
    equal = torch.equal(a, b)
    if equal:
        return {'equal': True, 'count_a': int(a.numel()), 'count_b': int(b.numel()), 'missing': 0, 'extra': 0}
    aset = set(a.tolist())
    bset = set(b.tolist())
    return {
        'equal': False,
        'count_a': int(a.numel()),
        'count_b': int(b.numel()),
        'missing': len(aset - bset),
        'extra': len(bset - aset),
        'missing_sample': sorted(list(aset - bset))[:8],
        'extra_sample': sorted(list(bset - aset))[:8],
    }

def qef_error_metrics(cpu, other):
    occ = compare_occ(cpu, other)
    if not occ['equal']:
        return {'occ_equal': False, **occ}
    rows = {'occ_equal': True, 'count': int(cpu['keys'].numel())}
    for name in ['mean_sum', 'cnt', 'qefs']:
        diff = (other[name] - cpu[name]).float()
        ref = cpu[name].float()
        rows[f'{name}_max_abs'] = float(diff.abs().max().item()) if diff.numel() else 0.0
        rows[f'{name}_rmse'] = float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0
        rows[f'{name}_rel_l2'] = float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0
    rows['intersected_mismatch'] = int((other['intersected'] != cpu['intersected']).sum().item())
    return rows

def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)

## Run four QEF implementations and compare occupancy

In [9]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

qef_outputs = {}
qef_outputs['cpu'] = ext.intersect_qef_cpu(triangles_cpu, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)
qef_outputs['old'] = ext.intersect_qef_old(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)
qef_outputs['ref'] = ext.intersect_qef_ref(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)
qef_outputs['new'] = ext.intersect_qef(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)
torch.cuda.synchronize()

canon = {name: canonicalize_qef(out) for name, out in qef_outputs.items()}
occ_rows = []
for name in ['old', 'ref', 'new']:
    row = {'method': name, **compare_occ(canon['cpu'], canon[name]), 'unique': canon[name]['unique']}
    occ_rows.append(row)
show_rows(occ_rows)

assert canon['cpu']['unique'], 'cpu produced duplicate voxel keys'
for name in ['old', 'ref', 'new']:
    assert canon[name]['unique'], f'{name} produced duplicate voxel keys'
    assert compare_occ(canon['cpu'], canon[name])['equal'], f'{name} occupancy differs from cpu'

,method,equal,count_a,count_b,missing,extra,unique
0,old,True,4676307,4676307,0,0,True
1,ref,True,4676307,4676307,0,0,True
2,new,True,4676307,4676307,0,0,True


## Direct `intersect_occ` API checks

`ref` currently has no standalone `intersect_occ_ref`; for ref occupancy we use `intersect_qef_ref(...)[0]` above. Old and new direct occ APIs are checked below.

In [10]:
occ_direct = {
    'cpu_occ': canonicalize_occ(ext.intersect_occ_cpu(triangles_cpu, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
    'old_occ': canonicalize_occ(ext.intersect_occ_old(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
    'new_occ': canonicalize_occ(ext.intersect_occ(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)),
}
torch.cuda.synchronize()

occ_api_rows = []
for name, occ in occ_direct.items():
    occ_api_rows.append({'method': name, **compare_occ(canon['cpu'], occ)})
show_rows(occ_api_rows)

for name, occ in occ_direct.items():
    assert compare_occ(canon['cpu'], occ)['equal'], f'{name} differs from cpu qef occupancy'

,method,equal,count_a,count_b,missing,extra
0,cpu_occ,True,4676307,4676307,0,0
1,old_occ,True,4676307,4676307,0,0
2,new_occ,True,4676307,4676307,0,0


## QEF error estimates against CPU

In [11]:
qef_rows = []
for name in ['old', 'ref', 'new']:
    qef_rows.append({'method': name, **qef_error_metrics(canon['cpu'], canon[name])})
show_rows(qef_rows)

,method,occ_equal,count,mean_sum_max_abs,mean_sum_rmse,mean_sum_rel_l2,cnt_max_abs,cnt_rmse,cnt_rel_l2,qefs_max_abs,qefs_rmse,qefs_rel_l2,intersected_mismatch
0,old,True,4676307,0.000006,8.521194e-08,3.210950e-08,0.0,0.0,0.0,0.000028,1.620905e-07,9.370419e-08,0
1,ref,True,4676307,0.000006,7.726928e-08,2.911659e-08,0.0,0.0,0.0,0.000029,1.992649e-07,1.153819e-07,0
2,new,True,4676307,0.000006,7.112312e-08,2.680056e-08,0.0,0.0,0.0,0.000029,1.960446e-07,1.135162e-07,0


## GPU timing and memory: cold vs warm

Cold runs call `torch.cuda.empty_cache()` before measurement. Warm runs keep the allocator warm and report median over repeated measurements. Memory is peak allocated delta over the baseline after inputs are already resident on GPU.

In [12]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }

def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }

gpu_qef_methods = {
    'old': lambda: ext.intersect_qef_old(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    'ref': lambda: ext.intersect_qef_ref(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    'new': lambda: ext.intersect_qef(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
}
gpu_occ_methods = {
    'old': lambda: ext.intersect_occ_old(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
    'ref': lambda: ext.intersect_qef_ref(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES)[0],
    'new': lambda: ext.intersect_occ(triangles_cuda, voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES),
}

WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
bench_rows = []
for family, methods in [('occ', gpu_occ_methods), ('qef', gpu_qef_methods)]:
    for name, fn in methods.items():
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        warm = summarize_records(warm_records)
        bench_rows.append({
            'family': family,
            'method': name,
            'cold_ms': cold['ms'],
            'cold_peak_alloc_mb': cold['peak_alloc_mb'],
            'cold_peak_reserved_mb': cold['peak_reserved_mb'],
            **warm,
        })

show_rows(bench_rows)

,family,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,occ,old,65.026848,289.055664,8.0,55.107151,54.169601,289.055664,0.0
1,occ,ref,92.787712,1961.154297,1280.0,91.583485,90.411102,1961.154297,0.0
2,occ,new,10.019840,121.776367,2.0,9.049088,8.904704,121.776367,0.0
3,qef,old,108.351746,2973.064941,3708.0,108.152321,107.619331,2972.628906,0.0
4,qef,ref,97.459198,1961.154297,1280.0,90.853088,88.163330,1961.154297,0.0
5,qef,new,19.619841,380.900391,2.0,18.355216,17.726463,380.900391,0.0
